# Ford Car Price Prediction 🚗
This notebook explores Ford vehicle data, performs exploratory data analysis, and trains a baseline Linear Regression model to estimate car prices.

## 1. Import libraries and load data

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Support both a local project folder and Google Colab's common path.
local_csv = Path('ford.csv')
colab_csv = Path('/content/ford.csv')
csv_path = local_csv if local_csv.exists() else colab_csv

if not csv_path.exists():
    raise FileNotFoundError('Could not find ford.csv. Place it beside this notebook or at /content/ford.csv.')

df = pd.read_csv(csv_path)
df.head()

## 2. Inspect the dataset

In [ ]:
print(f'Dataset shape: {df.shape}')
display(df.info())
display(df.describe(include='all').T)
display(df.isnull().sum().sort_values(ascending=False))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='price', bins=30, kde=True)
plt.title('Distribution of Ford Car Prices')
plt.xlabel('Price')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Between Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='model', y='price')
plt.xticks(rotation=90)
plt.title('Car Price by Model')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='transmission', y='price', ax=axes[0])
axes[0].set_title('Price by Transmission')
axes[0].tick_params(axis='x', rotation=30)
sns.boxplot(data=df, x='fuelType', y='price', ax=axes[1])
axes[1].set_title('Price by Fuel Type')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Prepare features and target
The target is `price`. Numerical columns are standardized and categorical columns are one-hot encoded inside a preprocessing pipeline. Keeping preprocessing inside the pipeline prevents information from the test set leaking into the training process.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

target = 'price'
X = df.drop(columns=[target]).copy()
y = df[target].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Numerical features:', numeric_features)
print('Categorical features:', categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 5. Evaluate the model
R² and Adjusted R² summarize model fit; MAE and RMSE express prediction errors in the target's units. Scores are calculated from the current dataset when this notebook runs.

In [ ]:
r2 = r2_score(y_test, y_pred)
n = len(y_test)
p = model.named_steps['preprocessor'].transform(X_test).shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1)) if n > p + 1 else np.nan
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

results = pd.DataFrame({
    'Metric': ['R² Score', 'Adjusted R² Score', 'MAE', 'RMSE'],
    'Value': [r2, adjusted_r2, mae, rmse],
)
display(results)

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], linestyle='--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Ford Car Prices')
plt.tight_layout()
plt.show()

## 6. Conclusion and next steps
This notebook provides a baseline price-prediction workflow. Next, compare additional regression models, use cross-validation, inspect residuals and outliers, and tune the modelling approach. Model quality should be judged using the evaluation metrics above rather than assumed in advance.